In [1]:
import gc
import os
import time
import random
import warnings
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
import multiprocessing as mp
from itertools import groupby
import plotly.graph_objs as go #visualization library
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import matplotlib.dates as mdates
import plotly.figure_factory as ff
from collections import defaultdict
from tqdm.notebook import tqdm as tqdm
from plotly.subplots import make_subplots
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
warnings.filterwarnings("ignore")
%matplotlib inline
from datetime import datetime

In [2]:
def forward_imputation(data):
    store_index = []
    for index in range(len(data)):
        try:
            if len(data) > 1:
                store_index.append([data[index], data[index + 1]])
            else:
                store_index.append(data[index])
        except IndexError as ix:
            continue
    return store_index


def _forward_with_last_measured(data, hours_data=None):
    cols_data = data.columns.to_list()
    times_hours_data = np.arange(0, hours_data, 1)
    notna_cols_indexes, nan_cols_indexes = defaultdict(list), defaultdict(list)
    indexes_last_indicator = defaultdict(list)
    for col in cols_data:
        notna_cols_indexes[col].append(list(data.loc[pd.notna(data[col]), :].index))
        nan_cols_indexes[col].append(list(data.loc[pd.isna(data[col]), :].index))
        indexes_last_indicator[col].append([data[col].notna()[::-1].idxmax(),
                                            data[col].isna()[::-1].idxmax()])
    notna_cols_indexes_ = {key: list(itertools.chain.from_iterable(value))
                           for key, value in notna_cols_indexes.items()
                           if list(itertools.chain.from_iterable(value))}
    notna_cols_indexes_ = {key: value for key, value in notna_cols_indexes_.items() if
                           len(value) != len(times_hours_data)}

    nan_cols_indexes = {key: list(itertools.chain.from_iterable(value))
                        for key, value in nan_cols_indexes.items()}
    nan_cols_indexes_ = {key: value for key, value in nan_cols_indexes.items()
                         if len(value) != len(times_hours_data)}
    nan_cols_indexes_ = {key: value for key, value in nan_cols_indexes_.items() if value}

    indexes_last_indicator = {key: list(itertools.chain.from_iterable(value))
                              for key, value in indexes_last_indicator.items()}
    matrix_indexes_notna = {key: forward_imputation(value) for key, value in notna_cols_indexes_.items()}
    matrix_notna_with_last = {
        key: list(itertools.chain.from_iterable([matrix_indexes_notna[key], [indexes_last_indicator[key]]]))
        for key in indexes_last_indicator if key in matrix_indexes_notna}
    final_matrix_indexes = defaultdict(list)
    for key, vals in matrix_notna_with_last.items():
        for val in vals:
            if isinstance(val, list):
                final_matrix_indexes[key].append(val)
    matrix_range_indexes_cols = {key: [final_matrix_indexes[key], nan_cols_indexes_[key]]
                                 for key in nan_cols_indexes_ if key in final_matrix_indexes}

    range_indexes_cols_imputed = {}
    for key, value in matrix_range_indexes_cols.items():
        range_values = [[notna_ind, nan] for nan in value[1]
                        for notna_ind in value[0] if notna_ind[0] <= nan <= notna_ind[1]]
        range_indexes_cols_imputed[key] = range_values
    return range_indexes_cols_imputed


def forward_with_last_measured_value_with_time_elasped_interval(data_copy, mask_data, hrs_used=None):
    results = _forward_with_last_measured(data_copy, hours_data=hrs_used)
    data = data_copy.copy()
    mask_forward = mask_data.copy()
    for key, values in results.items():
        for k, group in groupby(values, lambda x: x[0]):
            vals_ = list(itertools.chain.from_iterable(list(group)))
            vals_ = np.array([val for val in vals_ if not isinstance(val, list)])
            if k[1] < hrs_used - 1:
                for index in np.arange(k[0] + 1, k[1]):
                    data.at[index, key] = data._get_value(k[0], key)
                    mask_forward.at[index, key] = index - k[0]
            else:
                for index in range(int(k[0]) + 1, int(k[1]) + 1):
                    if pd.isnull(data._get_value(index, key)):
                        data.at[index, key] = data._get_value(k[0], key)
                        mask_forward.at[index, key] = index - k[0]
                    else:
                        pass
    return data, mask_forward

In [3]:
def is_subject_folder(x):
    return str.isdigit(x)

In [4]:
all_features= ['ALP', 'ALT', 'AST', 'Age', 'Albumin', 'BUN', 'Bilirubin', 'Cholesterol', 
               'Creatinine', 'DiasABP', 'FiO2', 'GCS', 'Gender', 'Glucose', 'HCO3', 'HCT', 'HR', 
               'Height', 'ICUType', 'K', 'Lactate', 'MAP', 'MechVent', 'Mg', 'NIDiasABP', 'NIMAP', 
               'NISysABP', 'Na', 'PaCO2', 'PaO2', 'Platelets', 'RespRate', 'SaO2', 'SysABP', 'Temp', 
               'TropI', 'TropT', 'Urine', 'WBC', 'Weight', 'pH']

critical_features = list(set(['Age', 'GCS', 'HR', 'MAP', 'DiasABP', 'SysABP', 'NIDiasABP', 'NIMAP', 
                    'RespRate', 'Temp', 'PaCO2', 'Pao2_Fio2', 'BUN', 'Creatinine', 'Lactate', 'Glucose',
                    'Albumin', 'Bilirubin', 'PaCO2', 'HCO3', 'pH', 'Platelets','K','Na','Urine','WBC','MechVent']))
len(all_features) , len(critical_features)

(41, 26)

In [5]:
# Helper function to handle lists
cols_min = ['Albumin','HCO3','HCT','Mg','K','PaCO2','PaO2','pH','Platelets','Na','WBC',
            'DiasABP','GCS','HR','MAP','SaO2','SysABP','NIDiasABP','NIMAP','NISysABP']
cols_sum = ['Urine']
def aggregate_list(column, func):
    return column.apply(lambda x: func(x) if isinstance(x, list) else x)


def count_unique(column):
    return column.apply(lambda x: len(set(x)) if isinstance(x, list) else pd.Series([x]).nunique())


def convert_events_to_timeseries(events, variable_column='Parameter', value_column='Value',
                                 aggr_min_var=cols_min, aggr_sum_var=cols_sum):
    metadata = events[['Time']].sort_values(by=['Time']).drop_duplicates(keep='first').set_index('Time')
    timeseries = events[['Time', variable_column, value_column]].groupby(['Time', 
                                                                        variable_column])[value_column].apply(list).reset_index()
    #timeseries = events[['Time', variable_column]].groupby(['Time', variable_column])[value_column].apply(list).reset_index()

    timeseries = timeseries.pivot(index='Time', columns=variable_column, values=value_column) \
        .merge(metadata, left_index=True, right_index=True).sort_index(axis=0).reset_index()
    episodes_subjects = timeseries.copy()
    #del timeseries["Time"]
    timeseries['Time'] = timeseries['Time'].str.split(':').str[0]
    timeseries['Time'] = timeseries['Time'].astype(float)
    aggr_max_colums = list(set(timeseries.columns.tolist()[1:]) - set(aggr_min_var) - set(aggr_sum_var))
    
    # print(timeseries.columns.tolist()[1:])
    episodes_all = timeseries.copy()
    data_features_stats = timeseries.copy()
    cols = timeseries.columns.tolist()[1:]
    cols_min = set(aggr_min_var) - (set(aggr_min_var) - set(cols))
    cols_sum = set(aggr_sum_var) - (set(aggr_sum_var) - set(cols))

    for col in aggr_max_colums:
        episodes_all[col] = aggregate_list(episodes_all[col], np.nanmax)
    for col in cols_sum:
        episodes_all[col] = aggregate_list(episodes_all[col], np.nansum)
    for col in cols_min:
        episodes_all[col] = aggregate_list(episodes_all[col], np.nanmin)

    all_cols = [column for column in episodes_all.columns.tolist() if column != 'Time']
    for col in all_cols:
        data_features_stats[col] = count_unique(data_features_stats[col])

    features_stats = {feat: ["max"] for feat in data_features_stats.columns.to_list() if feat != "Time"}
    grouped = data_features_stats.groupby(['Time'], as_index=False, dropna=False)
    stats_data = pd.concat([grouped[col].agg(agg_func).rename(columns={col: f'{col}_{agg_func}'})
                            for col, agg_funcs in features_stats.items() for agg_func in agg_funcs],
                           axis=1)
    stats_data = stats_data.loc[:, ~stats_data.columns.duplicated()]
    del stats_data["Time"]
    return episodes_subjects, timeseries, episodes_all

In [6]:
def discretized_events_to_timeseries(timeseries_data, interval_max_data=None, variables=all_features, 
                                     aggr_min_var=cols_min, aggr_sum_var=cols_sum):
    
    timeseries = timeseries_data.copy()
    del timeseries["RecordID"]
    all_columns = timeseries.columns.to_list()
    features_measured_columns = [column for column in all_columns if column != 'Time']
    # Add supplementary variable if not measured with value nan
    additional_columns = list(set(variables) - set(features_measured_columns))
    additional_values = list(itertools.repeat(np.nan, timeseries.shape[0]))
    # Create a DataFrame with additional columns filled with NaN values
    additional_data = pd.DataFrame({col: additional_values for col in additional_columns})
    # Concatenate the new columns with the original DataFrame
    timeseries = pd.concat([timeseries, additional_data], axis=1)
    # Optional: To de-fragment the DataFrame, create a copy
    timeseries.set_index('Time', inplace=True)
    timeseries = timeseries.copy()
    # interval sampling for regular timeseries
    interval_sampling = np.arange(0, interval_max_data, 1)
    # present hours helpers measured
    present_hours_data = timeseries.index.values.tolist()
    # adding missing hours with value nan if the helpers has not been measured
    new_indexes = [hour for hour in interval_sampling if hour not in present_hours_data]
    # value nan for variables which has not been measured
    range_vals = list(itertools.repeat(np.nan, len(timeseries.columns.tolist())))
    new_data = pd.DataFrame(columns=timeseries.columns.tolist())
    for index in new_indexes:
        new_series = pd.Series(dict(zip(timeseries.columns.tolist(), range_vals)), name=index)
        new_data = pd.concat([new_data, new_series.to_frame().T])
    episode_timeseries = pd.concat([timeseries, new_data]).sort_index()
    episode_timeseries.index.rename('Time', inplace=True)
    episode_timeseries.reset_index(inplace=True)

    # differents type of aggregation based on variables
    aggr_max_colums = list(
        set(list(set(episode_timeseries.columns.tolist()[1:]) - set(aggr_min_var))) - set(aggr_sum_var))
    agg_max_colums = dict.fromkeys(aggr_max_colums, np.nanmax)
    agg_sum_colums = dict.fromkeys(aggr_sum_var, np.nansum)
    agg_min_colums = dict.fromkeys(aggr_min_var, np.nanmin)
    # aggregates functions applied on each variable
    aggregates_func_var = {**agg_max_colums, **agg_sum_colums, **agg_min_colums}

    episode_timeseries = episode_timeseries.groupby(["Time"],as_index=False,
                                                    dropna=False).agg(aggregates_func_var)
    episode_timeseries.set_index('Time', inplace=True)
    features_all = episode_timeseries.columns.to_list()
    for feature_used in features_all:
        episode_timeseries.loc[episode_timeseries[feature_used] == 0, feature_used] = np.nan
    
    episode_timeseries = episode_timeseries.astype(float)
    episode_timeseries["Pao2_Fio2"] = episode_timeseries.apply(lambda e: round(100 * (e['PaO2'] / e['FiO2']), 2), axis=1)
    episode_timeseries = episode_timeseries[critical_features]
    episode_timeseries['Age'] = episode_timeseries['Age'].bfill().ffill()
    episode_timeseries = episode_timeseries.loc[:, episode_timeseries.columns.notna()]
    episode_timeseries = episode_timeseries.reindex(sorted(episode_timeseries.columns), axis=1)
    return episode_timeseries

In [7]:
dn_data="Documents/physionet.org/files/challenge-2012/1.0.0/phase1/timeseries"
dn_targets="Documents/physionet.org/files/challenge-2012/1.0.0/phase1/targets"

In [8]:
hrs_data_max=47

In [9]:
output_dir=f"events_{hrs_data_max+1}_hours_data_for_all_subjects"
# From pickle
import pickle
with open('/Documents/physionet.org/files/challenge-2012/1.0.0/phase1/valid_subjects.pkl', 'rb') as f:
    valid_subjects = pickle.load(f)
len(valid_subjects)

11988

In [10]:
for outcome in os.listdir(dn_targets):
    
    timeseries_path = os.listdir(dn_data)
    ts_target_dn = [ts_dn for ts_dn in os.listdir(dn_data) 
                    if ts_dn.endswith(outcome.split(".")[0][-2:])][0]
    
    subjects_outs = pd.read_csv(os.path.join(dn_targets, outcome), sep=',')
    
    for event in tqdm(os.listdir(os.path.join(dn_data,ts_target_dn)), 
                      total=len(os.listdir(os.path.join(dn_data,ts_target_dn))),
                      desc=f"Processing subject's event data"):
        #print("event.split('.')[0]", event.split('.')[0])
        if int(event.split('.')[0]) in  valid_subjects:
            subject_event = pd.read_csv(os.path.join(os.path.join(dn_data,ts_target_dn), event), sep=',')
            subject_all_events, all_events, episodes = convert_events_to_timeseries(subject_event)
            
            episodes_timeseries = discretized_events_to_timeseries(episodes, interval_max_data=hrs_data_max + 1)
            episodes_timeseries_masked = np.where((pd.isnull(episodes_timeseries.values)), 0, 1)
            # masking vector for episode timeseries for generating time delta
            masked_timeseries = np.where((pd.isnull(episodes_timeseries.values)), np.nan, 0)
            masked_episode_timeseries = pd.DataFrame(masked_timeseries,
                                                     columns=episodes_timeseries.columns.to_list())
            ep_timeseries_for_imp, time_elasped_episode_timeseries = forward_with_last_measured_value_with_time_elasped_interval(
                episodes_timeseries,
                masked_episode_timeseries,
                hrs_used=hrs_data_max + 1)
            statics_data = subjects_outs[subjects_outs["RecordID"]==episodes["RecordID"].values[0]]
            dn = os.path.join(os.path.join(os.path.dirname(os.path.abspath(dn_targets)), output_dir),
                              event.split(".")[0])
            #print(dn)
            if not os.path.exists(dn):
                os.makedirs(dn)
            # episodes_timeseries
            episodes_timeseries.to_csv(os.path.join(dn, f"{str(event.split('.')[0])}_episodes_timeseries.csv"), 
                                       index=False)
            np.save(os.path.join(dn, f"{str(event.split('.')[0])}_episodes_timeseries.npy"), 
                    episodes_timeseries.values)
            ep_timeseries_for_imp.to_csv(os.path.join(dn, f"{str(event.split('.')[0])}_episodes_timeseries_imputated.csv"),
                                         index=False)
            np.save(os.path.join(dn, f"{str(event.split('.')[0])}_episodes_timeseries_imputated.npy"), ep_timeseries_for_imp.values)
            # time delta for episode timeseries
            time_elasped_episode_timeseries.to_csv(os.path.join(dn, f"{str(event.split('.')[0])}_time_elasped_episodes_timeseries.csv"),
                                                   index=False)
            np.save(os.path.join(dn, f"{str(event.split('.')[0])}_time_elasped_episodes_timeseries.npy"),
                    time_elasped_episode_timeseries.values)
            
            statics_data["In-hospital_death"].to_csv(os.path.join(dn, f"{str(event.split('.')[0])}_targets.csv"), 
                                                     index=False)
            np.save(os.path.join(dn, f"{str(event.split('.')[0])}_targets.npy"), 
                    statics_data["In-hospital_death"].values)
            
            statics_data[["SAPS-I", "SOFA"]].to_csv(os.path.join(dn, f"{str(event.split('.')[0])}_static_variables.csv"), 
                                                     index=False)
            np.save(os.path.join(dn, f"{str(event.split('.')[0])}_static_variables.npy"), 
                    statics_data[["SAPS-I", "SOFA"]].values)
            
              

Processing subject's event data:   0%|          | 0/4000 [00:00<?, ?it/s]

Processing subject's event data:   0%|          | 0/4000 [00:00<?, ?it/s]

Processing subject's event data:   0%|          | 0/4000 [00:00<?, ?it/s]

In [11]:
selected_hours_data = 48
def all_subjects_data_cohorts(root_subjects_data, output_dir):
    subdirectories = os.listdir(root_subjects_data)
    subjects = list(filter(is_subject_folder, subdirectories))
    ID_subjects, all_statics_data, all_targets = [], [], []
    all_timeseries_data, all_timeseries_imp_data = [], []
    all_mask_data, all_delta_time_data, all_clinical_notes = [], [], []
    for subject in tqdm(subjects, desc='Iterating over subjects'):
        ID_subjects.append(subject)
        subject_path = os.path.join(root_subjects_data, subject)
        # load statics variables
        statics = np.load(os.path.join(subject_path, f"{subject}_static_variables.npy"),allow_pickle=True)
        # load targets helpers
        targets = np.load(os.path.join(subject_path, f"{subject}_targets.npy"),allow_pickle=True)
        # load timeseries helpers
        episodes_timeseries = np.load(os.path.join(subject_path, f"{subject}_episodes_timeseries.npy"),
                                      allow_pickle=True)
        episodes_timeseries_imputated = np.load(
            os.path.join(subject_path, f"{subject}_episodes_timeseries_imputated.npy"), allow_pickle=True)
        # load delta time helpers
        delta_time_data = np.load(os.path.join(subject_path, f"{subject}_time_elasped_episodes_timeseries.npy"),
                                  allow_pickle=True)

        all_statics_data.append(statics)
        all_targets.append(targets)
        
        if episodes_timeseries.shape[0]!=48 or episodes_timeseries.shape[1]!=41:
            all_timeseries_data.append(episodes_timeseries[:48])
            all_timeseries_imp_data.append(episodes_timeseries_imputated[:48])
            all_delta_time_data.append(delta_time_data[:48])
        else:
            all_timeseries_data.append(episodes_timeseries)
            all_timeseries_imp_data.append(episodes_timeseries_imputated)
            all_delta_time_data.append(delta_time_data)
            #print(subject, episodes_timeseries.shape, delta_time_data.shape, episodes_timeseries[:48].shape)
    dn = os.path.join(os.path.dirname(os.path.abspath(root_subjects_data)), output_dir)
    if not os.path.exists(dn):
        os.makedirs(dn)
    np.savez(os.path.join(dn, f"subjects_data.npz"), 
             statics_data=np.stack(all_statics_data),
             targets_data=np.stack(all_targets), 
             timeseries_data=np.stack(all_timeseries_data),
             timeseries_imp_data=np.stack(all_timeseries_imp_data),
             delta_time_data=np.stack(all_delta_time_data))


In [12]:
root_subjects_data=f"/Documents/physionet.org/files/challenge-2012/1.0.0/phase1/{output_dir}"

In [13]:
all_subjects_data_cohorts(root_subjects_data, f"all_subjects_data_{hrs_data_max+1}_hours")

Iterating over subjects:   0%|          | 0/11988 [00:00<?, ?it/s]